In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from glob import glob
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.amp import autocast, GradScaler
from matplotlib import pyplot as plt

----- Load data -----

In [ ]:
# Lấy đường dẫn data
def load_data_paths(base_path):
    train_imgs  = sorted(glob(os.path.join(base_path, "Train", "**", "Image", "*.jpg")))
    train_masks = sorted(glob(os.path.join(base_path, "Train", "**", "Mask", "*.png")))
    test_imgs   = sorted(glob(os.path.join(base_path, "Test", "**", "Image", "*.jpg")))

    return train_imgs, train_masks, test_imgs

In [ ]:
ROOT_PATH = '/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation'

# check root_path
for root, dirs, files in os.walk(ROOT_PATH):
    print(root, len(files))
    break

train_imgs, train_masks, test_imgs = load_data_paths(ROOT_PATH)

print(f'Size train dataset images: {len(train_imgs)}')
print(f'Size train dataset masks: {len(train_masks)}')
print(f'Size test dataset images: {len(test_imgs)}')

In [ ]:
# vissualize check sample
img = Image.open(train_imgs[0])
mask = Image.open(train_masks[0])

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Image")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Mask")
plt.show()

----- Preprocessing + Augmentation -----

In [ ]:
# ----------------- Tính Mean/Std -----------------
def get_mean_std(image_paths, save_path="mean_std.json"):
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            stats = json.load(f)
        return np.array(stats["mean"]), np.array(stats["std"])

    n_images = 0
    mean_sum = np.zeros(3)
    sq_mean_sum = np.zeros(3)

    for path in tqdm(image_paths, desc="Computing mean/std"):
        img = cv2.imread(path, cv2.IMREAD_COLOR).astype(np.float32) / 255.0
        mean_sum += img.mean(axis=(0, 1))
        sq_mean_sum += (img ** 2).mean(axis=(0, 1))
        n_images += 1

    mean = mean_sum / n_images
    mean_of_sq = sq_mean_sum / n_images
    std = np.sqrt(mean_of_sq - mean ** 2)

    stats = {"mean": mean.tolist(), "std": std.tolist()}
    with open(save_path, "w") as f:
        json.dump(stats, f, indent=4)

    return mean, std

In [ ]:
print(os.listdir('/kaggle/working'))

In [ ]:
# ! rm -rf /kaggle/working/pos_weight.json submission submission.csv checkpoint.pth best_model.pth .virtual_documents training_log.csv state.db

----------------- Hàm xử lý và tăng cường dữ liệu -----------------

In [ ]:
img_size = (512, 512)

# 🔹 Tính/lấy mean-std
mean, std = get_mean_std(train_imgs, save_path="mean_std.json")
print(mean, std)

# 🔹 Dùng trong Albumentations
train_transform = A.Compose([
    A.Resize(*img_size),
    # Biến đổi hình học
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), rotate=(-45, 45), translate_percent=(0.1, 0.1), p=0.5, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomSizedCrop(min_max_height=(400, 512), size=img_size, p=0.5),
    
    # Biến đổi quang học và độ méo
    A.OneOf([
        A.ElasticTransform(alpha=50, sigma=5, p=1),
        A.GridDistortion(p=1),
        A.OpticalDistortion(p=1)
    ], p=0.5),

    # Biến đổi màu sắc và ánh sáng
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.CLAHE(p=0.5),
    A.RandomGamma(p=0.5),
    A.ToGray(p=0.1),

    # Thêm nhiễu và các vùng thiếu thông tin
    A.OneOf([
        A.GaussNoise(p=1),
        A.ISONoise(p=1),
        A.GridDropout(p=1)
    ], p=0.2),

    # Các phép biến đổi khác
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(*img_size),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
])


----- Create dataset & dataloader -----

In [ ]:
# ----- SegmentationDataset -----
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Đọc ảnh trực tiếp ở dạng màu (3 kênh)
        image = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Albumentations expects RGB
        mask = None

        if self.mask_paths is not None:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            # Chia mask cho 255 để chuẩn hóa về khoảng [0, 1]
            mask = mask.astype(np.float32) / 255.0

        if self.transform is not None:
            if mask is not None:
                augmented = self.transform(image=image, mask=mask)
                image = augmented["image"]
                # đảm bảo mask là float
                mask = augmented["mask"].float()
            else:
                augmented = self.transform(image=image)
                image = augmented["image"]

        return (image, mask) if mask is not None else image

In [ ]:
# ----- Create dataset & dataloader -----
train_imgs_split, val_imgs, train_masks_split, val_masks = train_test_split(
    train_imgs, train_masks, test_size=0.2, random_state=42
)

train_dataset = SegmentationDataset(train_imgs_split, train_masks_split, transform=train_transform)
val_dataset   = SegmentationDataset(val_imgs, val_masks, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
first_batch_images, first_batch_masks = next(iter(train_loader))
print("Kích thước của mask trong DataLoader:", first_batch_masks.shape)
print("Giá trị pixel tối thiểu của mask:", first_batch_masks.min().item())
print("Giá trị pixel tối đa của mask:", first_batch_masks.max().item())
print("Kiểu dữ liệu của mask:", first_batch_masks.dtype)

# In ra một số giá trị cụ thể để kiểm tra
print("Giá trị 3x3 của mask:", first_batch_masks[0, 100:103, 100:103])

--- Vissualize data after augmentation ---

In [ ]:
def denormalize_img(img, mean, std):
    """
    Denormalizes a tensor image.
    """
    img = img.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

In [ ]:
def visualize_batch(loader, title, mean, std):
    """
    Trực quan hóa một batch hình ảnh và mask.
    """
    batch_images, batch_masks = next(iter(loader))
    
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    fig.suptitle(title, fontsize=16)
    
    for i in range(4):
        img_np = denormalize_img(batch_images[i], mean, std)
        mask_np = batch_masks[i].squeeze().cpu().numpy()
        
        # Plot image
        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f'Image {i+1}')
        axes[0, i].axis('off')
        
        # Plot mask
        axes[1, i].imshow(mask_np, cmap='gray')
        axes[1, i].set_title(f'Mask {i+1}')
        axes[1, i].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
# 1. Trực quan hóa một batch dữ liệu đã được tăng cường
visualize_batch(train_loader, "Trực quan hóa một batch dữ liệu đã được Augmentation", mean, std)

----- calculate_pos_weight (mức độ mất cân bằng lớp) -----

In [ ]:
# ----- calculate_pos_weight function -----
def calculate_pos_weight(loader):
    """
    Tính toán pos_weight (tỷ lệ giữa số pixel background và forgeground) 
    từ tập dữ liệu huấn luyện.
    """
    total_pos_pixels = 0
    total_neg_pixels = 0
    
    for _, masks in tqdm(loader, desc="Tính pos_weight"):
        total_pos_pixels += torch.sum(masks > 0).item()
        total_neg_pixels += torch.sum(masks == 0).item()
        
    if total_pos_pixels == 0:
        return 1.0 # Trả về 1 nếu không có pixel dương tính, tránh chia cho 0
    
    return total_neg_pixels / total_pos_pixels

In [ ]:
def get_pos_weight(loader):
    """
    Tải pos_weight từ file nếu tồn tại, ngược lại tính toán và lưu lại.
    """
    save_path = "pos_weight.json"
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            pos_weight_val = json.load(f)["pos_weight"]
            print(f"Đã tìm thấy file '{save_path}'. Tải giá trị pos_weight: {pos_weight_val:.2f}")
            return pos_weight_val
    else:
        print("Không tìm thấy file pos_weight.json, đang tiến hành tính toán...")
        pos_weight_val = calculate_pos_weight(loader)
        with open(save_path, "w") as f:
            json.dump({"pos_weight": pos_weight_val}, f)
        print(f"Đã tính toán và lưu giá trị pos_weight: {pos_weight_val:.2f} vào file '{save_path}'")
        return pos_weight_val

In [ ]:
def visualize_pos_weight(pos_weight):
    """
    Trực quan hóa mức độ mất cân bằng lớp.
    """
    labels = ['Background', 'Skin']
    values = [1, 1/pos_weight]
    
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(labels, values, color=['#1f77b4', '#ff7f0e'])
    ax.set_title('Tỷ lệ Pixel: Nền vs. Da')
    ax.set_ylabel('Giá trị Tương đối')
    ax.set_xticks(labels)
    ax.tick_params(axis='x', rotation=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for i, v in enumerate(values):
        ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# 2. Trực quan hóa mức độ mất cân bằng lớp
pos_weight_value = get_pos_weight(train_loader)
visualize_pos_weight(pos_weight_value)

----------------- Xây dựng Model UNet -----------------

In [ ]:
# ----- ConvBlock (residual + GroupNorm + Dropout) -----
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.3) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(8, out_channels)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(8, out_channels)
        self.relu2 = nn.ReLU(inplace=True)

        self.dropout = nn.Dropout2d(p=dropout)

        # projection nếu in_channels != out_channels
        self.res_conv = None
        if in_channels != out_channels:
            self.res_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x):
        identity = x
        out = self.relu1(self.gn1(self.conv1(x)))
        out = self.relu2(self.gn2(self.conv2(out)))
        out = self.dropout(out)

        if self.res_conv is not None:
            identity = self.res_conv(identity)

        return out + identity

In [ ]:
# ----- Encoder -----
class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = ConvBlock(in_channels, out_channels)

    def forward(self, x):
        x = self.pool(x)
        return self.block(x)

In [ ]:
# ----- Decoder -----
class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        # input của ConvBlock phải là tổng số kênh sau khi concatenate
        self.block = ConvBlock(out_channels * 2, out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        x = self.conv1x1(x)

        # Đảm bảo cùng kích thước khi concat
        diffY = skip.size()[2] - x.size()[2]
        diffX = skip.size()[3] - x.size()[3]
        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])

        x = torch.cat([skip, x], dim=1)
        return self.block(x)

In [ ]:
# ----- Unet -----
class UNet(nn.Module):
    def __init__(self, n_channels, n_classes) -> None:
        super().__init__()
        self.in_conv = ConvBlock(n_channels, 64)

        self.enc1 = Encoder(64, 128)
        self.enc2 = Encoder(128, 256)
        self.enc3 = Encoder(256, 512)
        self.enc4 = Encoder(512, 1024)

        # Sửa channel mismatch
        self.dec1 = Decoder(1024, 512)
        self.dec2 = Decoder(512, 256)
        self.dec3 = Decoder(256, 128)
        self.dec4 = Decoder(128, 64)

        self.out_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.in_conv(x)  # 64
        x2 = self.enc1(x1)  # 128
        x3 = self.enc2(x2)  # 256
        x4 = self.enc3(x3)  # 512
        x5 = self.enc4(x4)  # 1024

        x = self.dec1(x5, x4)
        x = self.dec2(x, x3)
        x = self.dec3(x, x2)
        x = self.dec4(x, x1)

        return self.out_conv(x)

----------------- Hàm mất mát và đánh giá -----------------

In [ ]:
# ----- DiceLoss -----
class DiceLoss(nn.Module):
    """
    Dice Loss for binary segmentation.
    This loss function is useful for handling class imbalance.
    """
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        """
        Calculates the Dice Loss.
        Args:
            logits (torch.Tensor): Raw outputs from the model (e.g., [B, 1, H, W]).
            targets (torch.Tensor): Ground truth masks (e.g., [B, H, W] or [B, 1, H, W]).
        Returns:
            torch.Tensor: The mean Dice Loss over the batch.
        """
        # Ensure targets have the same dimensions as logits for broadcasting
        if targets.dim() == 3:
            targets = targets.unsqueeze(1).float()

        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Flatten the tensors for easier calculation
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        # Calculate intersection and union
        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        # Calculate Dice coefficient
        dice_coeff = (2. * intersection + self.smooth) / (union + self.smooth)

        # Dice Loss is 1 - Dice coefficient
        dice_loss = 1. - dice_coeff.mean()

        return dice_loss

In [ ]:
# # ----- BCEDiceLoss -----
# class BCEDiceLoss(nn.Module):
#     def __init__(self, pos_weight=None, **kwargs):
#         super().__init__()
#         self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#         self.dice = DiceLoss(**kwargs)

#     def forward(self, logits, targets):
#         if targets.dim() == 3:
#             targets = targets.unsqueeze(1)
            
#         bce_loss = self.bce(logits, targets)
#         dice_loss = self.dice(logits, targets)
#         return bce_loss + dice_loss

In [ ]:
# ----- FocalLoss -----
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2, reduction='mean', **kwargs):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        if targets.dim() == 3:
            targets = targets.unsqueeze(1)
        
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss

In [ ]:
# ----- BCEDiceFocalLoss -----
class BCEDiceFocalLoss(nn.Module):
    def __init__(self, pos_weight=None, focal_alpha=0.8, focal_gamma=2, **kwargs):
        super().__init__()
        # Sử dụng Focal Loss thay cho BCE
        self.focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, reduction='mean')
        self.dice = DiceLoss(**kwargs)
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        if targets.dim() == 3:
            targets = targets.unsqueeze(1)
        
        # Áp dụng pos_weight thủ công cho Focal Loss nếu cần
        focal_loss = self.focal(logits, targets)
        if self.pos_weight is not None:
            # Tự động tính toán lại trọng số để áp dụng pos_weight cho Focal Loss
            # Đây là một cách đơn giản, có thể tinh chỉnh thêm
            pos_term = targets * self.pos_weight
            focal_loss = torch.mean(focal_loss * (1 + pos_term))

        dice_loss = self.dice(logits, targets)
        return focal_loss + dice_loss

In [ ]:
# ----- dice_score -----
def dice_score(preds, targets, eps=1e-6):
    if targets.ndim == 3:
        targets = targets.unsqueeze(1)
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float()
    intersection = (preds_bin * targets).sum(dim=(2,3))
    union = preds_bin.sum(dim=(2,3)) + targets.sum(dim=(2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

In [ ]:
# ----- iou_score -----
def iou_score(preds, targets, eps=1e-6):
    if targets.ndim == 3:
        targets = targets.unsqueeze(1)
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float()
    intersection = (preds_bin * targets).sum(dim=(2,3))
    union = (preds_bin + targets - preds_bin * targets).sum(dim=(2,3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


In [ ]:
# ----- Evaluate function -----
def evaluate(model, loader, device, criterion=None):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            if criterion is not None:
                loss = criterion(outputs, targets)
                total_loss += loss.item() * inputs.size(0)

            total_dice += dice_score(outputs, targets) * inputs.size(0)
            total_iou += iou_score(outputs, targets) * inputs.size(0)

    n = len(loader.dataset)
    mean_loss = total_loss / n if criterion is not None else 0.0
    mean_dice = total_dice / n
    mean_iou = total_iou / n
    return mean_loss, mean_dice, mean_iou

----- Train model -----

In [ ]:
# ----- Train function -----
def train_model(model, pos_weight, train_loader, val_loader, device, epochs=50, lr=1e-5, patience=12):
    # Tính pos_weight trước khi bắt đầu huấn luyện
    pos_weight = get_pos_weight(train_loader)
    pos_weight_tensor = torch.tensor([pos_weight], device=device)
    print(f"Giá trị pos_weight được tính toán: {pos_weight:.3f}")
    
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = BCEDiceFocalLoss(pos_weight=pos_weight_tensor)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    scaler = GradScaler()

    best_dice = 0.0
    counter = 0
    start_epoch = 0

    checkpoint_path = "checkpoint.pth"
    log_file_path = "training_log.csv"
    
    if os.path.exists(checkpoint_path):
        print(f"Tải checkpoint từ {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Tạo một OrderedDict mới và xử lý tiền tố 'module.' một cách an toàn
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in checkpoint['model_state_dict'].items():
            name = k[7:] if k.startswith('module.') else k  # Kiểm tra và loại bỏ tiền tố
            new_state_dict[name] = v
        
        # Tải state_dict đã sửa vào mô hình
        # Lưu ý: Nếu mô hình ban đầu không phải là DataParallel, nó sẽ tự động thêm tiền tố 'module.'
        # khi tải một state_dict không có tiền tố này.
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(new_state_dict)
        else:
            model.load_state_dict(new_state_dict)
            
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_dice = checkpoint['best_dice']
        counter = checkpoint['counter']
        print(f"Đã tải checkpoint thành công. Tiếp tục huấn luyện từ epoch {start_epoch + 1}.")
        
        # Đọc và in dữ liệu log cũ
        if os.path.exists(log_file_path):
            print("\n--- LỊCH SỬ HUẤN LUYỆN TRƯỚC ĐÓ ---")
            log_df = pd.read_csv(log_file_path)
            print(log_df.to_string())
            print("--------------------------------------\n")
        else:
            print("Không tìm thấy file log cũ. Bắt đầu ghi log mới.")
            with open(log_file_path, 'w') as f:
                f.write("epoch,train_loss,val_loss,val_dice,val_iou,lr\n")
    else:
        print("Không tìm thấy checkpoint. Bắt đầu huấn luyện từ đầu.")
        # Nếu không có checkpoint, tạo file log mới.
        if os.path.exists(log_file_path):
            os.remove(log_file_path)
            print(f"Đã xóa file log cũ: {log_file_path}")
        with open(log_file_path, 'w') as f:
            f.write("epoch,train_loss,val_loss,val_dice,val_iou,lr\n")
        print(f"Đã tạo file log mới: {log_file_path}")

    train_losses, val_losses, dice_scores, iou_scores = [], [], [], []

    for epoch in range(start_epoch + 1, epochs+1):
        model.train()
        epoch_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()

            with autocast(device_type=device):
                logits = model(images)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item() * images.size(0)

        epoch_loss /= len(train_loader.dataset)
        val_loss, val_dice, val_iou = evaluate(model, val_loader, device, criterion)

        train_losses.append(epoch_loss)
        val_losses.append(val_loss)
        dice_scores.append(val_dice)
        iou_scores.append(val_iou)

        scheduler.step(val_dice)

        print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")
        print(f"Epoch [{epoch}/{epochs}] Loss: {epoch_loss:.5f} | Val Dice: {val_dice:.5f} | Val IoU: {val_iou:.5f}")

        # Ghi log vào file
        log_data = f"{epoch},{epoch_loss:.5f},{val_loss:.5f},{val_dice:.5f},{val_iou:.5f},{optimizer.param_groups[0]['lr']}\n"
        with open(log_file_path, 'a') as f:
            f.write(log_data)
            
        # Lưu checkpoint sau mỗi epoch
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'counter': counter
        }
        torch.save(checkpoint, checkpoint_path)
        
        if val_dice > best_dice:
            best_dice = val_dice
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), "best_model.pth")
            else:
                torch.save(model.state_dict(), "best_model.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered")
                break

    if os.path.exists("best_model.pth"):
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(torch.load("best_model.pth"))
        else:
            model.load_state_dict(torch.load("best_model.pth"))
    return model, train_losses, val_losses, dice_scores, iou_scores

In [ ]:
# ----- Model Execution -----
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet(n_channels=3, n_classes=1).to(device)

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPU!")
    model = nn.DataParallel(model)

model, train_losses, val_losses, dice_scores, iou_scores = train_model(
    model, pos_weight_value, train_loader, val_loader, device, epochs=200, lr=1e-5, patience=12
)

----- Vissualization -----

In [ ]:
# Plot training curves
def plot_training_curves(train_losses, val_losses, dice_scores, iou_scores):
    epochs = range(1, len(train_losses) + 1)

    plt.style.use("ggplot")
    plt.figure(figsize=(14,5))

    # Loss
    plt.subplot(1,2,1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train & Validation Loss')
    plt.legend()
    plt.grid(True)

    # Metrics
    plt.subplot(1,2,2)
    plt.plot(epochs, dice_scores, 'g-', label='Dice Score')
    plt.plot(epochs, iou_scores, 'm-', label='IoU Score')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title('Dice & IoU Scores')
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
file_path = "training_log.csv"
if os.path.exists(file_path):
    df = pd.read_csv(file_path)

    train_losses = df['train_loss'].tolist()
    val_losses = df['val_loss'].tolist()
    dice_scores = df['val_dice'].tolist()
    iou_scores = df['val_iou'].tolist()

    plot_training_curves(train_losses, val_losses, dice_scores, iou_scores)
else:
    plot_training_curves(train_losses, val_losses, dice_scores, iou_scores)

In [ ]:
# ----------------- Hàm mã hóa RLE và tạo file nộp bài -----------------
def rle_encode(mask):
    '''
    mask: numpy array, 1 - mask, 0 - background
    returns: run-length encoded string
    '''
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

In [ ]:
def create_submission_file(output_dir, test_paths, submission_file="submission.csv"):
    print("Bắt đầu tạo file nộp bài...")
    submission_list = []
    for path in tqdm(test_paths, desc="Đang mã hóa RLE"):
        filename = os.path.basename(path).replace(".jpg", ".png")
        mask_path = os.path.join(output_dir, filename)
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            rle = rle_encode(mask)
            submission_list.append([filename, rle])

    submission_df = pd.DataFrame(submission_list, columns=['ID', 'Predicted_Mask'])
    submission_df.to_csv(submission_file, index=False)
    print(f"Đã tạo file nộp bài thành công: {submission_file}")

----------------- Dự đoán trên tập dữ liệu kiểm tra (chưa tta) -----------------

In [ ]:
# ----------------- Hàm dự đoán trên tập dữ liệu kiểm tra -----------------
def predict(model, test_paths, device, transform, output_dir="submission"):
    print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra...")
    os.makedirs(output_dir, exist_ok=True)
    model.eval()
    
    test_dataset = SegmentationDataset(test_paths, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4)

    with torch.no_grad():
        for i, images in tqdm(enumerate(test_loader), total=len(test_loader), desc="Đang dự đoán"):
            images = images.to(device)
            outputs = model(images)
            
            # Chuyển đổi logits thành mask nhị phân (0 hoặc 255)
            preds = torch.sigmoid(outputs)
            preds = (preds > 0.5).float() * 255
            
            for j in range(preds.size(0)):
                # Lấy tên file gốc
                original_path = test_paths[i * test_loader.batch_size + j]
                filename = os.path.basename(original_path)
                
                # Lưu mask
                mask_np = preds[j].squeeze().cpu().numpy().astype(np.uint8)
                output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
                cv2.imwrite(output_path, mask_np)
    print(f"Đã lưu các mask dự đoán vào thư mục '{output_dir}'")

In [ ]:
# Dự đoán trên tập dữ liệu kiểm tra sau khi huấn luyện
predict(model, test_imgs, device, val_transform)

In [ ]:
# Tạo file nộp bài sau khi dự đoán
create_submission_file(output_dir="submission", test_paths=test_imgs)

In [ ]:
import os
import cv2
import glob
import matplotlib.pyplot as plt

def plot_img_mask(submission_folder):
    # Kiểm tra xem thư mục có tồn tại và có chứa file nào không
    if not os.path.exists(submission_folder) or not glob.glob(os.path.join(submission_folder, "*.png")):
        print("Thư mục 'submission' không tồn tại hoặc không chứa file mask nào.")
    else:
        # Lấy đường dẫn của file mask
        first_mask_path = sorted(glob.glob(os.path.join(submission_folder, "*.png")))[6]
        img_path = test_imgs[6]
        
        # Đọc mask bằng OpenCV
        # OpenCV đọc ảnh theo định dạng BGR, nhưng mask chỉ có một kênh màu xám
        img = Image.open(first_mask_path)
        mask = Image.open(img_path)
        
        # Hiển thị mask
        plt.figure(figsize=(8, 8))
        plt.subplot(1, 2, 1)
        plt.imshow(mask, cmap='gray')
        plt.title(f"Visualizing Mask: {os.path.basename(first_mask_path)}")
        plt.axis('off')
        plt.show()
    
        plt.subplot(1, 2, 2)
        plt.imshow(img)
        plt.title(f"Visualizing Mask: {os.path.basename(img_path)}")
        plt.axis('off')
        plt.show()
    print("Đã hoàn thành việc hiển thị mask.")


In [ ]:
# Đường dẫn đến thư mục chứa các mask dự đoán
submission_folder = "submission"
plot_img_mask(submission_folder)

In [ ]:
print(os.listdir('/kaggle/working'))

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta) -----------------

In [ ]:
# ----------------- Hàm dự đoán trên tập dữ liệu kiểm tra -----------------
def post_process_mask(mask, kernel_size=5, min_area=1000):
    """
    Áp dụng hậu xử lý cho mask để làm mịn và loại bỏ nhiễu.
    """
    # Lấp đầy các lỗ nhỏ và làm mịn đường viền
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # Loại bỏ các vùng nhiễu nhỏ
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            cv2.drawContours(mask, [contour], -1, (0), -1)
            
    return mask

In [ ]:
# Hàm tìm ngưỡng tối ưu
def find_optimal_threshold(model, val_loader, device, num_thresholds=100):
    """
    Tìm ngưỡng tối ưu trên tập validation một cách tiết kiệm bộ nhớ.
    Sử dụng cache để lưu lại kết quả.
    """
    cache_file = "optimal_threshold_results.json"
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            results = json.load(f)
            best_threshold = results["optimal_threshold"]
            best_dice = results["dice_score"]
            print(f"Đã tìm thấy file '{cache_file}'. Tải kết quả từ lần chạy trước.")
            print(f"Ngưỡng tối ưu đã lưu: {best_threshold:.3f}, Dice Score tương ứng: {best_dice:.5f}")
            return best_threshold, best_dice

    print("Bắt đầu tìm ngưỡng tối ưu...")
    model.eval()
    thresholds = np.linspace(0.01, 0.99, num_thresholds)
    best_dice = 0.0
    best_threshold = 0.5
    
    # Lưu kết quả Dice Score của từng ngưỡng vào một dictionary
    threshold_results = {threshold: 0.0 for threshold in thresholds}
    
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="Đang tính toán và tìm ngưỡng"):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)

            if targets.dim() == 3:
                targets = targets.unsqueeze(1)

            for threshold in thresholds:
                preds_bin = (probs > threshold).float()
                
                # Tính toán Dice Score cho batch hiện tại
                intersection = (preds_bin * targets).sum(dim=(2, 3))
                union = preds_bin.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
                dice_per_image = (2. * intersection + 1e-6) / (union + 1e-6)
                
                # Tích lũy tổng Dice Score cho ngưỡng này
                threshold_results[threshold] += dice_per_image.sum().item()

    # Tính Dice Score trung bình cho từng ngưỡng
    n_images = len(val_loader.dataset)
    for threshold, total_dice in threshold_results.items():
        avg_dice = total_dice / n_images
        print(f"Ngưỡng: {threshold:.3f}, Dice Score: {avg_dice:.5f}")

        if avg_dice > best_dice:
            best_dice = avg_dice
            best_threshold = threshold
            
    print(f"\nNgưỡng tối ưu tìm được trên tập validation là: {best_threshold:.3f}")
    print(f"Dice Score tương ứng là: {best_dice:.5f}")
    
    # Lưu kết quả vào file
    with open(cache_file, "w") as f:
        json.dump({"optimal_threshold": best_threshold, "dice_score": best_dice}, f, indent=4)
        
    return best_threshold, best_dice

In [ ]:
# def predict(model, test_paths, device, transform, output_dir="submission"):
#     print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra...")
#     os.makedirs(output_dir, exist_ok=True)
#     model.eval()
    
#     test_dataset = SegmentationDataset(test_paths, transform=transform)
#     test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4)

#     with torch.no_grad():
#         for i, images in tqdm(enumerate(test_loader), total=len(test_loader), desc="Đang dự đoán"):
#             images = images.to(device)
#             outputs = model(images)
            
#             preds = torch.sigmoid(outputs)
            
#             for j in range(preds.size(0)):
#                 original_path = test_paths[i * test_loader.batch_size + j]
#                 filename = os.path.basename(original_path)
                
#                 mask_np = preds[j].squeeze().cpu().numpy()
#                 mask_np = (mask_np > 0.5).astype(np.uint8) * 255
                
#                 # Áp dụng hậu xử lý
#                 processed_mask = post_process_mask(mask_np)
                
#                 output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
#                 cv2.imwrite(output_path, processed_mask)
#     print(f"Đã lưu các mask dự đoán vào thư mục '{output_dir}'")

In [ ]:
# Hàm dự đoán sử dụng Test-Time Augmentation
def predict_with_tta(model, test_paths, device, original_transform, output_dir="submission_tta", threshold=0.5):
    print("Bắt đầu dự đoán trên tập dữ liệu kiểm tra với TTA...")
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    # Định nghĩa các phép biến đổi TTA và các phép biến đổi ngược tương ứng
    tta_transforms = [
        # (hàm biến đổi, hàm đảo ngược)
        (lambda img: img, lambda mask: mask),  # Gốc
        (lambda img: cv2.flip(img, 1), lambda mask: cv2.flip(mask, 1)), # Lật ngang
        (lambda img: cv2.flip(img, 0), lambda mask: cv2.flip(mask, 0)), # Lật dọc
        (lambda img: cv2.flip(img, -1), lambda mask: cv2.flip(mask, -1)), # Lật ngang và dọc
        (lambda img: cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)), # Xoay 90 độ
        (lambda img: cv2.rotate(img, cv2.ROTATE_180), lambda mask: cv2.rotate(mask, cv2.ROTATE_180)), # Xoay 180 độ
        (lambda img: cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE), lambda mask: cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)), # Xoay 270 độ
    ]
        
    with torch.no_grad():
        for original_path in tqdm(test_paths, desc="Đang dự đoán với TTA"):
            image_raw = cv2.imread(original_path, cv2.IMREAD_COLOR)
            image_raw = cv2.cvtColor(image_raw, cv2.COLOR_BGR2RGB)
            
            tta_masks = []
            
            for transform, inverse_transform in tta_transforms:
                # Áp dụng TTA
                augmented_image = transform(image_raw)

                # Chuẩn hóa và chuyển đổi sang tensor
                transformed_input = original_transform(image=augmented_image)["image"].unsqueeze(0).to(device)
                
                # Dự đoán
                pred_mask = torch.sigmoid(model(transformed_input)).squeeze().cpu().numpy()
                
                # Áp dụng phép biến đổi ngược
                inverted_mask = inverse_transform(pred_mask)
                
                tta_masks.append(inverted_mask)

            # Hợp nhất các dự đoán bằng cách tính trung bình
            final_mask = np.mean(tta_masks, axis=0)
            final_mask = (final_mask > threshold).astype(np.uint8) * 255
            
            # Áp dụng hậu xử lý
            processed_mask = post_process_mask(final_mask)
            
            filename = os.path.basename(original_path)
            output_path = os.path.join(output_dir, filename.replace(".jpg", ".png"))
            cv2.imwrite(output_path, processed_mask)
    
    print(f"Đã lưu các mask dự đoán với TTA vào thư mục '{output_dir}'")

In [ ]:
# Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
predict_with_tta(model, test_imgs, device, val_transform, output_dir="submission_tta", threshold=0.5)

In [ ]:
# Tạo file nộp bài sau khi dự đoán
create_submission_file(output_dir="submission_tta", test_paths=test_imgs, submission_file="submission_tta.csv")

In [ ]:
# Đường dẫn đến thư mục chứa các mask dự đoán
submission_folder = "submission_tta"
plot_img_mask(submission_folder)

----------------- Dự đoán trên tập dữ liệu kiểm tra (đã tta - best thredshold) -----------------

In [ ]:
best_threshold, best_dice_val = find_optimal_threshold(model, val_loader, device)

In [ ]:
# Dự đoán và tạo file nộp bài sử dụng TTA và Hậu xử lý
predict_with_tta(model, test_imgs, device, val_transform, output_dir="submission_tta_best_thredshold", threshold=best_threshold)

In [ ]:
# Tạo file nộp bài sau khi dự đoán
create_submission_file(output_dir="submission_tta_best_thredshold", test_paths=test_imgs, submission_file="submission_tta_best_thredshold.csv")

In [ ]:
# Đường dẫn đến thư mục chứa các mask dự đoán
submission_folder = "submission_tta_best_thredshold"
plot_img_mask(submission_folder)